# slm-toolbox — Quickstart

A tour of the SLM control SDK: generate phase patterns, preview them and their far-field beam
inline, compose hybrid holograms, and run an automated measurement — **all with no hardware**.

Install first (from the project folder): `pip install -e .`

In [ ]:
import slm_toolbox as slm
from slm_toolbox import patterns, compose, show, far_field_image
print("slm-toolbox", slm.__version__)

SHAPE = (512, 512)
WAVELENGTH_M = 1064e-9
PIXEL_PITCH_M = 8e-6

## 1. A single vortex (OAM) hologram

`show(...)` returns a `PIL.Image`, which Jupyter renders inline.

In [ ]:
show(patterns.vortex_phase(SHAPE, l=3))

## 2. Predict the actual beam it makes — no hardware

`far_field_image` simulates the focal-plane diffraction pattern. A vortex → a doughnut with a dark
core.

In [ ]:
far_field_image(patterns.vortex_phase(SHAPE, l=3),
                wavelength_m=WAVELENGTH_M, pixel_pitch_m=PIXEL_PITCH_M, waist_m=60e-6)

## 3. A forked hologram (vortex + blazed grating)

Additive composition: sum phase terms, wrap to [0, 2π). The grating steers the OAM beam into the +1
diffraction order.

In [ ]:
fork = compose.sum_phases(
    patterns.vortex_phase(SHAPE, l=3),
    patterns.blazed_grating_phase(SHAPE, period_px=12),
)
show(fork)

## 4. A hybrid beam: vortex + lens + steering

Any number of phase structures compose in one hologram.

In [ ]:
hybrid = compose.sum_phases(
    patterns.vortex_phase(SHAPE, l=2),
    patterns.fresnel_lens_phase(SHAPE, focal_length_m=1.0,
                                wavelength_m=WAVELENGTH_M, pixel_pitch_m=PIXEL_PITCH_M),
    patterns.blazed_grating_phase(SHAPE, period_px=20, angle_deg=45),
)
show(hybrid)

## 5. Drive an SLM + instruments — with mocks (no hardware)

The `SLM` class is the scripting entry point. Swap the mock instruments for the real
`PM100` / `HikrobotCamera` / Elliptec drivers when you're at the bench — the API is identical.

In [ ]:
from slm_toolbox import SLM
from slm_toolbox.instruments import mock
import numpy as np

with SLM(wavelength_nm=1064) as s:
    # mock power meter whose reading models the calibration grating's diffraction efficiency
    def signal():
        g = s._last_gray
        return 0.0 if g is None else float(np.sin(np.pi * int(g.max()) / 255.0) ** 2)
    pm = mock.MockPM100(wavelength_nm=1064, signal_fn=signal, noise=1e-4)

    # fully automated gray->phase (Method C) calibration, driven by the (mock) power meter:
    curve = s.run_efficiency_calibration(measure_fn=pm.measure_fn(averages=3), settle_s=0.0)
    print(s.calibration_status())

    # a measurement sweep: vortex charge vs. reading
    for l in range(-2, 3):
        s.display_phase(s.vortex(l), s.grating(period_px=12), settle_s=0.0)
        print(f"l={l:+d}  reading={pm.get_power():.4g} W")

## Next

- Real hardware: replace `mock.MockPM100(...)` with `PM100(visa_resource)` and
  `mock.MockHikrobotCamera(...)` with `HikrobotCamera(...)`; open an `SLM()` on the HDMI display.
- Camera-feedback wavefront self-calibration: see `slm_toolbox.autocalibrate.optimize_zernike`
  and `examples/self_calibration_example.py`.
- Calibrate for your wavelength: see `calibration_SOP.md`.